# Riskfolio-Lib Tutorial:
<br><a href="https://www.kqzyfj.com/click-101360347-15150084?url=https%3A%2F%2Flink.springer.com%2Fbook%2F9783031843037" target="_blank">
<div>
<img src="https://raw.githubusercontent.com/dcajasn/Riskfolio-Lib/refs/heads/master/docs/source/_static/Button.png" height="40" />
</div>
<br>
</a>
<a href="https://www.paypal.com/ncp/payment/GN55W4UQ7VAMN" target="_blank">
<div>
<img src="https://raw.githubusercontent.com/dcajasn/Riskfolio-Lib/refs/heads/master/docs/source/_static/Button2.png" height="40" />
</div>
</a>

<br><a href='https://ko-fi.com/B0B833SXD' target='_blank'><img height='36' style='border:0px;height:36px;' src='https://cdn.ko-fi.com/cdn/kofi1.png?v=2' border='0' alt='Buy Me a Coffee at ko-fi.com' /></a>
<br>
<br>__[Financionerioncios](https://financioneroncios.wordpress.com)__
<br>__[Orenji](https://www.linkedin.com/company/orenj-i)__
<br>__[Riskfolio-Lib](https://riskfolio-lib.readthedocs.io/en/latest/)__
<br>__[Dany Cajas](https://www.linkedin.com/in/dany-cajas/)__

## Tutorial 54: Black Litterman with Adanos Market Sentiment Views

This notebook shows how to use an external sentiment API as an optional source of analyst views for the Black Litterman model. The API request is kept in an independent notebook function, and the notebook includes sample sentiment data so the optimization remains reproducible without an API key.


## 1. Loading historical prices

The example uses the local `assets_data.csv` file included with Riskfolio-Lib. This keeps the portfolio optimization reproducible and independent from the sentiment API.


In [ ]:
import json
import os
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd
import riskfolio as rp
from IPython.display import display

pd.options.display.float_format = '{:.4%}'.format

assets = ["AAPL", "MSFT", "NVDA", "JPM", "XOM", "UNH", "PG", "HD", "BAC", "PFE"]
assets.sort()

data = pd.read_csv("assets_data.csv", index_col="Dates", parse_dates=True)
prices = data.loc[:, assets].dropna()
returns = prices.pct_change().dropna()

display(returns.head())


## 2. Creating an optional Adanos sentiment request

Set `ADANOS_API_KEY` to use the live API. Without a key, the notebook uses the sample sentiment table below. The helper is intentionally local to the notebook; it is not part of Riskfolio-Lib's core calculation code.


In [ ]:
def adanos_compare_sentiment(
    tickers,
    source="news",
    days=7,
    api_key=None,
    base_url="https://api.adanos.org",
    timeout=10,
):
    """Download ticker-level sentiment from Adanos Market Sentiment API."""

    valid_sources = {"reddit", "x", "news", "polymarket"}
    source = source.strip().lower()
    if source not in valid_sources:
        raise ValueError("source must be one of: reddit, x, news, polymarket")

    tickers = [str(ticker).strip().upper() for ticker in tickers if str(ticker).strip()]
    if len(tickers) == 0:
        raise ValueError("tickers must contain at least one ticker")

    api_key = (api_key or os.getenv("ADANOS_API_KEY", "")).strip()
    if not api_key:
        raise ValueError("Please provide api_key or set ADANOS_API_KEY")

    query = urllib.parse.urlencode({"tickers": ",".join(tickers), "days": days})
    url = f"{base_url.rstrip('/')}/{source}/stocks/v1/compare?{query}"
    request = urllib.request.Request(
        url,
        headers={"X-API-Key": api_key, "Accept": "application/json"},
        method="GET",
    )

    with urllib.request.urlopen(request, timeout=timeout) as response:
        payload = json.loads(response.read().decode("utf-8"))

    sentiment = pd.DataFrame(payload.get("stocks", []))
    if sentiment.empty:
        return pd.DataFrame(index=pd.Index([], name="ticker"))
    sentiment["ticker"] = sentiment["ticker"].astype(str).str.upper()
    return sentiment.set_index("ticker")


In [ ]:
sample_sentiment = pd.DataFrame(
    {
        "ticker": assets,
        "sentiment_score": [0.28, 0.35, -0.18, 0.12, 0.16, 0.22, -0.08, 0.31, -0.14, 0.10],
        "buzz_score": [42.0, 74.0, 38.0, 33.0, 49.0, 57.0, 29.0, 68.0, 36.0, 41.0],
    }
).set_index("ticker")

api_key = os.getenv("ADANOS_API_KEY")
if api_key:
    sentiment = adanos_compare_sentiment(assets, source="news", days=7, api_key=api_key)
else:
    sentiment = sample_sentiment

display(sentiment)


## 3. Translating sentiment into Black Litterman views

The conversion below maps each sentiment score to an annual absolute return view. The scale is a modelling assumption and should be calibrated by the user.


In [ ]:
def sentiment_to_black_litterman_views(
    sentiment,
    assets,
    score_column="sentiment_score",
    annual_return_scale=0.05,
    min_abs_score=0.05,
):
    """Convert ticker sentiment scores into Black Litterman absolute views."""

    sentiment = sentiment.copy()
    sentiment.index = sentiment.index.astype(str).str.upper()

    P = []
    Q = []
    view_labels = []
    for i, asset in enumerate(assets):
        if asset not in sentiment.index:
            continue
        score = sentiment.loc[asset, score_column]
        if pd.isna(score) or abs(score) < min_abs_score:
            continue

        view = np.zeros(len(assets))
        view[i] = 1.0
        P.append(view)
        Q.append([float(score) * annual_return_scale])
        view_labels.append(asset)

    P = pd.DataFrame(P, columns=assets, index=view_labels)
    Q = pd.DataFrame(Q, columns=["views"], index=view_labels)
    return P, Q

P, Q = sentiment_to_black_litterman_views(
    sentiment,
    assets,
    annual_return_scale=0.05,
    min_abs_score=0.10,
)

display(P)
display(Q)


## 4. Estimating classic and Black Litterman portfolios


In [ ]:
port = rp.Portfolio(returns=returns)
port.assets_stats(method_mu="hist", method_cov="hist")

model = "Classic"
rm = "MV"
obj = "Sharpe"
hist = True
rf = 0
l = 0

w = port.optimization(model=model, rm=rm, obj=obj, rf=rf, l=l, hist=hist)
display(w.T)


In [ ]:
port.blacklitterman_stats(P.values, Q.values / 252, rf=rf, w=w, delta=None, eq=True)

w_bl = port.optimization(model="BL", rm=rm, obj=obj, rf=rf, l=l, hist=False)
display(w_bl.T)


## 5. Comparing portfolio weights


In [ ]:
weights = pd.concat([w, w_bl], axis=1)
weights.columns = ["Classic", "Black Litterman with Sentiment Views"]

display(weights)
